In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torchvision.transforms import transforms

In [3]:
transform = transforms.Compose([
    transforms.ToTensor()
])

trainset = MNIST(root = "./data",train =True ,download = True,transform = transform)
testset = MNIST(root="./data",train = False ,download =True , transform = transform)

100%|██████████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:07<00:00, 1.37MB/s]
100%|███████████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 111kB/s]
100%|███████████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:02<00:00, 824kB/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 1.80MB/s]


In [4]:
trainloader = DataLoader(trainset,shuffle = True , batch_size = 64)
testloader = DataLoader(testset,shuffle = True,batch_size = 64)

# Building CNN

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

        )


        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*128,256),nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) #flattening
        x = self.fc_layers(x)

        return x

In [6]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training and evaluation

In [10]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output,labels)
        loss.backward()

        optimizer.step() #update parameter
        epoch_training_loss += loss.item()

    model.eval()
    running_validation_loss = 0.0
    
    with torch.no_grad():
        for images,labels in testloader:
            output = model(images)
            loss = criterion(output,labels)
            running_validation_loss += loss.item()

    epoch_val_loss = running_validation_loss/ len(testloader)
    print(f"epoch{epoch+1}{epochs} and loss ={epoch_training_loss/len(trainloader)} and validation loss = {epoch_val_loss}")
    

epoch110 and loss =0.1703445478546014 and validation loss = 0.04546410209131516
epoch210 and loss =0.046360589575512944 and validation loss = 0.043470635382861944
epoch310 and loss =0.03297990276449685 and validation loss = 0.027657235322288788
epoch410 and loss =0.025691594041914415 and validation loss = 0.028124190958429596
epoch510 and loss =0.021273992388765005 and validation loss = 0.02682023326809604
epoch610 and loss =0.01648242992232864 and validation loss = 0.03381362921227612
epoch710 and loss =0.01362791974581103 and validation loss = 0.03465541592215382
epoch810 and loss =0.012066082193339012 and validation loss = 0.04220145683501388
epoch910 and loss =0.010066432441903892 and validation loss = 0.030744638628939905
epoch1010 and loss =0.009075995386300745 and validation loss = 0.030590390229431223


In [12]:
correct_labels = 0
total_labels = 0

with torch.no_grad():
    for images,labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
print(f"accuracy{correct_labels/total_labels *100}")

accuracy99.24


# Building RNN 


In [16]:
class RNN(nn.Module):
    def __init__(self,input_size=28,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size = input_size,hidden_size = hidden_size,num_layers=num_layers,batch_first=True)
        #fclayers
        self.fc = nn.Linear(hidden_size,10)


    def forward(self,x):
        out,_ = self.rnn(x)
        out = self.fc(out[:,-1,:])
        return out



        

In [18]:
model = RNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [21]:
epochs = 10
for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images,labels in trainloader:
        optimizer.zero_grad()
        images = images.squeeze(1)
    
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step() #update parameter
        epoch_training_loss += loss.item()

    model.eval()
    running_validation_loss = 0.0
    
    with torch.no_grad():
        for images,labels in testloader:
            images = images.squeeze(1)
            output = model(images)
            loss = criterion(output,labels)
            running_validation_loss += loss.item()

    epoch_train_loss = epoch_training_loss / len(trainloader)
    epoch_val_loss = running_validation_loss/ len(testloader)
   
    
    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Validation Loss: {epoch_val_loss:.4f}"
    )

Epoch 1/10 | Train Loss: 0.1852 | Validation Loss: 0.1710
Epoch 2/10 | Train Loss: 0.1659 | Validation Loss: 0.1525
Epoch 3/10 | Train Loss: 0.1530 | Validation Loss: 0.1379
Epoch 4/10 | Train Loss: 0.1450 | Validation Loss: 0.1613
Epoch 5/10 | Train Loss: 0.1296 | Validation Loss: 0.1248
Epoch 6/10 | Train Loss: 0.1269 | Validation Loss: 0.1572
Epoch 7/10 | Train Loss: 0.1173 | Validation Loss: 0.1544
Epoch 8/10 | Train Loss: 0.1151 | Validation Loss: 0.1440
Epoch 9/10 | Train Loss: 0.1142 | Validation Loss: 0.1201
Epoch 10/10 | Train Loss: 0.1049 | Validation Loss: 0.1115


In [24]:
with torch.no_grad():
    correct_vals = 0
    total_vals = 0 

    for images,labels in testloader:
        images = images.squeeze(1)

        outputs = model(images)
        _,predicted = torch.max(outputs,1)

        total_vals += labels.size(0)
        correct_vals += (predicted == labels).sum().item()
    accuracy = 100 * correct_vals/total_vals
    print(f"Accuracy= {accuracy:.2f}%")

Accuracy= 96.88%
